# RR BENCHMARK PHASE B1b: drug fix, row 8 donor fix, row 6 build, re-freeze (NO SCORING)
Fixes two B1 defects found before any score existed: (1) tacrolimus reference formula, (2) row 8 donor field is spelled `individuial` in the deposit. Builds row 6 from the spatial counts. Row 13 is not built (decision pending).
Run: hamburger > Runtime > Run all. Paste back everything between `=== PHASE B1b REPORT BEGIN ===` and `=== PHASE B1b REPORT END ===`.

In [ ]:
# CELL 1: setup (needs Phase A and B1 outputs on Drive)
from google.colab import drive
drive.mount("/content/drive")
import os,subprocess,sys
W="/content/drive/MyDrive/RitschelResearch/benchmark"; os.chdir(W)
subprocess.run([sys.executable,"-m","pip","install","-q","rdkit"],check=False)
import numpy as np,pandas as pd,gzip,re,urllib.request,io,hashlib,json,collections,tarfile,time
g=pd.read_csv("geneinfo_beta.txt",sep="\t")
LMG=g.loc[g.feature_space=="landmark"].copy(); LMG["gene_id"]=LMG.gene_id.astype(int)
genes=np.load("libA_genes.npy"); GI={int(x):i for i,x in enumerate(genes)}
SYM2E={s.upper():int(e) for e,s in zip(LMG.gene_id,LMG.gene_symbol)}
ENS2E={}
if "ensembl_id" in LMG.columns:
    for e,x in zip(LMG.gene_id,LMG.ensembl_id.astype(str)):
        for y in x.split("|"):
            if y.startswith("ENSG"): ENS2E[y.split(".")[0]]=int(e)
B1=json.load(open("signatures_frozen_B1.json"))
assert hashlib.sha256(open("signatures_frozen_B1.json","rb").read()).hexdigest()=="bb37a1aee429afe29d4a643bad665543f7d7fbadf5f8bc092d5dcfabefb3fbec","B1 FREEZE CHANGED"
SIG=B1["signatures"]
REPORT=[]; R=lambda *a: (print(*a), REPORT.append(" ".join(str(x) for x in a)))
R("B1 freeze verified; rows carried:",sorted(SIG))

In [ ]:
# CELL 2: drug formula check, CORRECTED RULE (addendum v1.2):
# reference = the formula shared by the MAJORITY of the drug's IDs; it must also equal the drug's known formula.
from rdkit import Chem
from rdkit.Chem.rdMolDescriptors import CalcMolFormula
KNOWN={"cladribine":"C10H12ClN5O3","tadalafil":"C22H19N3O4","doxycycline":"C22H24N2O8","hydroxyurea":"CH4N2O2",
       "ruxolitinib":"C17H18N6","dapagliflozin":"C21H25ClO6","empagliflozin":"C23H27ClO7","spironolactone":"C24H32O4S",
       "cyclosporin-a":"C62H111N11O12","tacrolimus":"C44H69NO12"}
ci=pd.read_csv("compoundinfo_beta.txt",sep="\t",dtype=str).fillna("")
D=pd.read_csv("drugs_resolved_phaseA.csv",dtype={"pert_id":str})
D["drug"]=D.query_name.str.replace("(skeleton)","",regex=False).replace({"cyclosporin a":"cyclosporin-a","cyclosporine":"cyclosporin-a","ciclosporin":"cyclosporin-a"})
def formula(pid):
    sm=[x for x in ci.loc[ci.pert_id==pid,"canonical_smiles"] if x]
    if not sm: return "NO_SMILES"
    m=Chem.MolFromSmiles(sm[0])
    if m is None: return "BAD_SMILES"
    return CalcMolFormula(max(Chem.GetMolFrags(m,asMols=True),key=lambda x:x.GetNumHeavyAtoms()))
D["formula"]=D.pert_id.map(formula); keep=[]
for (row,drug),grp in D.groupby(["row","drug"]):
    maj=grp.formula.value_counts().index[0]
    assert maj==KNOWN[drug], ("MAJORITY FORMULA != KNOWN",drug,maj,KNOWN[drug])
    for _,x in grp.iterrows():
        ok=(x.formula==maj) and not x.excluded and x.n_lib_sigs>0; keep.append(ok)
        R(f"row {row} {drug:15s} {x.pert_id} {x.formula:14s} majority {maj:14s} -> {'KEEP' if ok else 'DROP'}")
D["keep"]=keep; F=D[D.keep][["row","drug","pert_id","formula","n_lib_sigs","n_cells"]]
assert set(F[F.drug=="tacrolimus"].pert_id)=={"BRD-A96754982","BRD-K44094599","BRD-K65261396","BRD-K69608737"}, "TACROLIMUS SET UNEXPECTED"
for (row,drug),grp in F.groupby(["row","drug"]): assert grp.n_cells.max()>=2,("G2b",drug)
F.to_csv("drugs_frozen_v12.csv",index=False); R("drugs frozen v1.2:",len(F),"IDs over",F.groupby(['row','drug']).ngroups,"drugs")

In [ ]:
# CELL 3: shared helpers (statistic identical to control 2 and B1)
def stat(E,a,b,k=150):
    A=E[:,a]; B=E[:,b]; d=A.mean(1)-B.mean(1)
    pooled=np.sqrt(0.5*(A.var(1,ddof=1)+B.var(1,ddof=1)))
    fl=np.median(pooled[pooled>0]) if np.any(pooled>0) else 1.0
    st=d/np.maximum(pooled,fl*0.1); o=np.argsort(-st,kind="stable"); return o[:k].copy(),o[-k:].copy()
def build(row,series,E,grp,note):
    keep=~np.isnan(E).any(axis=1); idx=np.where(keep)[0]
    cases=[i for i,x in enumerate(grp) if x=="case"]; ctrls=[i for i,x in enumerate(grp) if x=="control"]
    assert len(cases)>0 and len(ctrls)>0, "EMPTY ARM IS FATAL"
    up,dn=stat(E[keep],cases,ctrls)
    SIG[row]=dict(series=series,n_case=len(cases),n_ctrl=len(ctrls),genes_usable=int(keep.sum()),
                  up=[int(genes[idx[i]]) for i in up],dn=[int(genes[idx[i]]) for i in dn],usable=[int(genes[i]) for i in idx],note=note)
    R(f"row {row} {series}: case units {len(cases)} control units {len(ctrls)} landmark genes usable {int(keep.sum())} | {note}")
def series_meta(s):
    meta=[]
    for line in open(s+"_series_matrix.txt",encoding="utf-8",errors="replace"):
        if line.startswith("!series_matrix_table_begin"): break
        meta.append(line)
    return meta
def fields_of(meta):
    gsm=[x.strip().strip('"') for x in [l for l in meta if l.startswith("!Sample_geo_accession")][0].rstrip("\n").split("\t")[1:]]
    F_=collections.OrderedDict()
    for l in meta:
        if l.startswith("!Sample_characteristics_ch1"):
            for i,v in enumerate(x.strip().strip('"') for x in l.rstrip("\n").split("\t")[1:]):
                if ":" in v: F_.setdefault(v.split(":",1)[0].strip().lower(),{})[i]=v.split(":",1)[1].strip()
    return gsm,F_
import base64
ROST_B64="H4sICAnCvmoCA3Jvc3QuY3N2AIWd3c4jRxVF73mWQfL5q3POA6DcgITEE6AwQkhAUIZIPD52rrodrz3KTVT76/a3V1WX1/S4Pd++/vyPr9++/P3bv778/eeffvnP7374yx/sEZFffvjLn/zYY86XH//67etvxxvG58uPP/37vz//9M/fRvv5kH3AuOGp1uGQgPGE8YLxwy8NxXdgfOlU9vhc3B4G486nCjgkYbxg/MB4w/jAOLc2aG3Q2hzGg18CWhu0tsOnao6gu31e5+ZQ3KG483Q7TLdDcYfiDtPt3NqhtUPrgNYBrQOmO6Dy8//oVw1oHdA6uHUMR1A8oXhC8YTiyes8YboTiiev84QLPGG6E1oXtC7czK2geMGMF894QfGCGS9oXdC6eFs7UPzAdB++ug8UPzDdp/hUUPxA8cPr/MCMNxRvnvGGGW8o3jzjDTPeULz5Gm+Y9IbiA8UHZnyg9UDr4dYDrcHWDGzN2NYMbM3A1myhNaiaLW9rYGsGtmZsawa2ZmBrxrbmYGsOtuZsaw625mBr/ig+1eGo4WwD459n3EHYHITNQdjcoDXYmoOtuR0Yx0vbQdUcVM1fqganAltzh9YefCooDrbmznPtMNdga+68zkHYHITNQdgchM1Z2ByEzUHYnIXNA4qDrXnyjIOwOQibs7A5CJuDsDkLm4OwOQibJ884OJsXFAdhcxA2Z2FzEDYHYfPiGQdn84IZB2FzEDY/0BpszcHW/EDlw3MNtuYHKoOqOaiaN1RufgsDVfOG1uBpDp7m7GkOnubgaT58aYOqOaiaD1/aA8VB1Xx4usHWfKA4qJqDqjmomoOqOauag6o5qJqzqjmomoOq+X5uHeBpAZ4WD4dxbB3gaQF31YIlLUDSAiQt+MZagKcFeFqApwV4WljyS0NxULVgVQtQtQBVC1a1AFULULVgVQtQtQBVC1a1AFULULVgVQtQtQBVi7jv55Hn+ab2zGLP80peEdZDhaZC9ZoVKkwVlgqPCluFo8LLErwnV5d4SwwTxyQwSUwKk4NJYzKYIINGBo0MGhk0Mmhk0MigkUEjg0YGjQwGGQwyGGQwyGCQwSCDQQaDDAYZDDJYZLDIYJHBIoNFBosMFhksMlhksMDgqSAPTAyTC4Pz/O/3P/z5j/W8qJ55PqquovApHh2vjK/q8Ck2Hevf/OoTn+LUcck47u89n37Cv/sT8d2fyO/+RH33J853f6K/+xPz3Z+4v4t3nuxX9sRgUSo8DxWaCl2FocJUYanwqLBVOCpUhFoRakWoFaFWhFoRakXo+p73ljQmg8lScn3Pe0sME8ckMElMChNkMMhgkMEgg0UGiwwWGSwyWGSwyGCRwSKDRQZLDPLxwMQwcUwCk8SkMDmYNCaDCTIwZGDIwJCBIQNDBoYMDBkYMjBkYMjAkYEjA0cGjgwcGTgycGTgyMCRgSODQAaBDAIZBDIIZBDIIJBBIINABoEMEhkkMkhkkMggkUEig0QGiQwSGSQyKGRQyKCQQSGDQgaFDAoZFDIoZFDI4CCDgwwOMjjI4CCDgwwOMjjI4CCDgwwaGTQyaGTQyKCRQSMD9MRET0z0xERPTPTERE9M9MRET0z0xERPTPTERE9M9MRET0z0xERPTPTERE9M9MRET0z0xERPTPTERE8s9MRCTyz0xEJPLPTEQk8s9MRCTyz0xEJPLPTEQk8s9MRCTywTf3r89c4Kh0eFrcJRofjzdaE5FppjoTkWmmOhORaaY6E5FppjoTkWmmOhORaaY6E5FppjoTkWmmOhORaaY6E5FppjoTkWmmOhORaaY6E5FppjoTkWmmOhORaaY6E5FppjoTkWmmOhORaaY6E5FppjoTkWmmOhORaaY6E5FppjqbuQpe5ClroLWeouZN3vQvrzZZ7XwzMce9TcP4V6D/fxUKGp0FUYKkwVlgqPCluFo0JFyBShyxvse+KYBCaJSWFyMGlMBpOl5PJ2+p4gA0cGjgwcGTgycGTgyMCRgSODQAaBDAIZBDIIZBDIIJBBIINABoEMEhkkMkhkkMggkUEig0QGiQwSGaTaFUrtCqX2zVL7Zql9s9S+WWrfLLVvlto3S+2bpQgdRegoQkcROorQUYSOInQUoaMIqffePYpQK0KtCLUi1IpQK0KtCLUi1IpQK0KtCI0iNIrQKEKjCI0iNIrQKEKjCI0iNIrQKkKrCK0itLgvL+7Li/vy4r68uC8v7ssL703n+kGZ98QwcUwCk8SkMDmYNCaDCTIwZGDIwJCBIQNDBoYMDBkYMjBkYMjAkYEjA0cGjgwcGTgycGTgyMCRgSODQAaBDAIZBDIIZBDIIJBBIINABoEMEhkkMkhkkMggkUEig0QGiQwSGSQyKGRQyKCQQSGDQgaFDAoZFDIoZFDI4CCDgwwOMjjI4CCDgwwOMjjI4CCDgwwaGTQyaGTQyKCRQSODRgaNDBoZNDIYZDDIYJDBIINBBoMMBhkMMhhkMMhgkcEig0UGiwwWGSw79Hn7rqT3sFU4KmSHPibugR4T90CPiXugx8Q90GPiHugxcQ/0mLgHegxd0tAlDV3S0CUNXdLQJQ1d0tAlDV3S0CUNXfL6HUtRL+9+BS9dbBifL1//9+M/f/nb17/9NtvPx1yY3ccNxh3GA8aTf6cLrfv4EcdAdxPdDbo7dHfo7tDdofvFt+/jxb+ri+4O3R3WiUPvgN5P08bXDuj+NG0+Rsx9CAYX376PtzgGGAQwSGCQMPcJ/VP0T5j/hHWf0Dth3hM6J3Qu6FzQuaBzwXov6FvQt8RaLzHXBb0Lej99Gs91oPvTp/kY6H+g/4H+B+b7wHwfsccd6N4w5w29G+a8oXOLa7yhd4t5b+jeMOcNvQd6D/Qe6D3Qe2CuBzoPzPWIdT5ivgd6r1jrC90Xui90X+i+0H3FfC/M98J8P82YznX98oP7uMG4w3jAeMJ4wfiB8YbxEd0+z3eAvwX4W4C/BfhbGHQGd7t+6cF9HDqD4wY4W4CzBTjb6+sOkKmHyFJk0F24W4C7Bbjb64sP8FzgbyH8LcDfImDeA+Y9oDs4WwT0Bl8L8LVI3tcCnC3A2UI4W4CzBThbpJjzbJGJaz3F3IPDBThcgMMFOFyAw0WV+J1g7oXDBTjc6yldPOZA9yPWvfC4AI97PZjLxwgOR6wFcLo4wAF87vUYLr4GOF2A072eveVzwToAp7s+cHsfh94t1j84XYDTBTjd60FbfA3wugCvC/C6AK+7PmJ7H4f5Bp+7Plx7H4fOKzqDzwX4XIDPXR+qvY9DZ3C5EC73eqCWs8/dE3wuwecSfC7B5xJ87vUYLf+uA8d8nu/XQ7R4LnC6BKdLcLoU9+QSvC7B6xK8LsHrErwuwesSvC7hXlzCvbiEe3EJLnd9WPY+Dn3B4xLuwSU43PUR2fs49AV/S/C3BH9L8LcEf3s9FItrKMR1LTwuweNeD8byMdAfPC7B4xLuvSXce0vhbwn33xLcLcHdEtwtwd0S3C3h/luCt6Xwtrx/wPIeiWkXt+BS6FveP1t5j0IclnxYicOOyJpPyUyOYHL/POU9Ekiakdw/SnmPxJ7fAsn9Y5T3iIkIu3s9JIvZiGVy//zkPRIbxDCT4VVy/+DkPWIi4mZeDi+S+ycmb5G4nfd6apYzXiQrrpsVq2QZibjBl8urZMUqWWRSQg1fT9Fy5iILkSX/KiUOO3xYi8OGD2Mm9uBIIDHnw4KjFGcsPoyB3J+prX7dHnpGr29emovdvgVBQVJQFBwKmoKhYCG4GO5bYBRQc6fmTs2dmjs1d2ru1NypeTx4du9fYfeWuchCZCmyEtkRmVih9y+me8uWsxRcUnBJwSUFlxRcUnBJwSUFlxRcUnApwaUElxJcSnApwaUElxJcSnApwaUElyO4HMHlCC5HcDmCyxFcjuByBJcjuBzBpQWXFlxacGnBpQWXFlxacGnBpQWXpk13BJQRUEZAGQFlBJQRUEZAGQFlBJQRi2UFlxVcVnBZwWUFlxVcVnBZwWUFl2Uub0/Wv2UmMhdZiCxFViI7ImuRjcgEFxNcTHAxwcUEFxNcTHAhhV1S2DVBxAQRF0RcEHFBxAURF0RcEHGxUlysFBLcJcG9Pmj/FhgFTkFQkBQUBbQiglZEUPOg5knNk5onNU9qntQ8qXlS86TmSc2Tmhc1L2pe1LyoeVHzouZFzYuaFzUvan6ouXDQFQ56/Scf3gLqL+xzhX2usM8V9rnCPlfY5wr7XGGfK+xzhX2usM8V9rnCPlfY59vT8PdMCOgKAV0hoCsEdIWArhDQFQK6QkBXCOgKAV0hoCsE9PrPRLwFdPksXT5L28fS9rG0fSxtH/tx+5jH9an3t8AocAqCgqSgKDgUNAU0/6+MyqNNvjITmYssRJYiK5ERDiMcn2+IvgJi4bQQXIBwWgtOa8FpLTitBafyaI2vjPo79Q/qH3QhBJUPKh9UPqh8UPmgmQ9qHtQ8qXlS86TmSc2Tmic1T2qe1DypeVLzouZFzYuaFzUval7UvKh5UfOi5kXNDzU/1PxQ80PNDzU/1PxQ80PNDzU/1LypeVPzpuZNzZuaNzVvat7UvKl5U/Oh5kPNh5oPNR9qPtR8qPlQ86HmQ82Xmi81X2q+1Hyp+VLzpeZLzZeak94Z6Z2R3hnpnZHeGemdkd4Z6Z2R3l0fQH8LqLlRc6PmRs2Nmhs1N2pOJmdkckYmZ2RyRiZnTs2dfdY8RHbz2dP267/99fr+4cu+fBteOuCyLd+GDQ/wzwcEHpCfDyg84Hw+oPGAz6UbS8/n0oOl53PpwdLzufRg6flcerD0fC49WHo/l14svZ9LX/6a5/+8jQXXO44AAA=="
_raw=gzip.decompress(base64.b64decode(ROST_B64)); assert hashlib.md5(_raw).hexdigest()=="ce6b1b2f36a4fff4c7034c4d99d4d280","ROSTER TRANSFER CORRUPTED"
RR=pd.read_csv(io.BytesIO(_raw)); ROS={(a,b):c for a,b,c in zip(RR.series,RR.gsm,RR.group)}

In [ ]:
# CELL 4: row 8 rebuilt with donor aggregation. The deposit's donor field is spelled "individuial".
s="GSE256539"; meta=series_meta(s); gsm,FF=fields_of(meta)
R("row 8 fields:",list(FF))
dkey=[k for k in FF if re.search(r"individ|patient|donor|subject",k)]
assert dkey, "NO DONOR FIELD"
dkey=dkey[0]
hdr=None; rows=[]; inb=False
for line in open(s+"_series_matrix.txt",encoding="utf-8",errors="replace"):
    if line.startswith("!series_matrix_table_end"): break
    if inb:
        p=line.rstrip("\n").split("\t")
        if hdr is None: hdr=[x.strip().strip('"') for x in p]
        else: rows.append(p)
    if line.startswith("!series_matrix_table_begin"): inb=True
samples=hdr[1:]; assert samples==gsm
acc={}
for p in rows:
    e=SYM2E.get(p[0].strip().strip('"').upper())
    if e is None or e not in GI: continue
    acc.setdefault(GI[e],[]).append(np.array([float(x.strip().strip('"')) for x in p[1:]]))
E=np.full((len(genes),len(samples)),np.nan)
for gi_,L in acc.items(): E[gi_]=np.median(np.vstack(L),axis=0)
# group from the B1 roster counts: re-derive from the same field B1's roster used (IPAH Lung / Control Lung)
grp=[ROS.get((s,x),"none") for x in samples]   # authoritative: the verified 2026-10-01 roster
cnt=collections.Counter(grp); R("row 8 per-ROI groups:",dict(cnt))
assert cnt["case"]==152 and cnt["control"]==68, "ROW 8 GROUPS DISAGREE WITH ROSTER (152/68)"
don=[FF[dkey].get(i,"NA") for i in range(len(samples))]
units=sorted(set((d,g_) for d,g_ in zip(don,grp) if g_ in("case","control")))
both=set(d for d,g_ in units if g_=="case")&set(d for d,g_ in units if g_=="control")
R(f"row 8 donor field '{dkey}': {len(set(don))} donors, units {len(units)}, donors in both arms {len(both)}")
Ea=np.column_stack([np.mean(E[:,[i for i,(d,g_) in enumerate(zip(don,grp)) if d==u and g_==gg]],axis=1) for u,gg in units])
build("8","GSE256539",Ea,[gg for u,gg in units],note=f"mean of {len(samples)} ROIs to donor via '{dkey}'")

In [ ]:
# CELL 5: row 6 spatial counts -> pseudobulk per GSM (library size on ALL genes) -> log2 CPM -> mean per donor within arm
s="GSE174647"; meta=series_meta(s); gsm,FF=fields_of(meta)
title=[x.strip().strip('"') for x in [l for l in meta if l.startswith("!Sample_title")][0].rstrip("\n").split("\t")[1:]]
T2I={t:i for i,t in enumerate(title)}
grp=[ROS.get((s,x),"none") for x in gsm]   # authoritative roster
ds=FF.get("disease state",{}); grp2=["case" if ds.get(i)=="MS" else "control" if ds.get(i)=="HI" else "none" for i in range(len(gsm))]
assert grp==grp2, "ROW 6 ROSTER DISAGREES WITH disease state FIELD"
cnt=collections.Counter(grp); R("row 6 GSM groups:",dict(cnt)); assert cnt["case"]==182 and cnt["control"]==28, "ROW 6 GROUPS DISAGREE WITH ROSTER"
f="GSE174647_ST_counts.tar.gz"; t0=time.time()
with tarfile.open(f) as tf:
    gl=pd.read_csv(tf.extractfile("ST_counts/genes.tsv"),sep="\t",header=None)
    bc=pd.read_csv(tf.extractfile("ST_counts/barcodes.tsv"),sep="\t",header=None)[0].astype(str).tolist()
R("row 6 genes.tsv cols",gl.shape[1],"first",gl.iloc[0].tolist(),"barcodes",len(bc),"first",bc[0])
sid=np.array([T2I.get(b.rsplit("_",1)[0],-1) for b in bc])
R("row 6 spots mapped to a GSM:",int((sid>=0).sum()),"of",len(bc),"; GSMs with spots:",len(set(sid[sid>=0].tolist())))
assert (sid>=0).mean()>0.99, "SPOT->GSM MAPPING FAILED"
nG=len(gl); nS=len(gsm); P=np.zeros(nG*nS)
with tarfile.open(f) as tf:
    fh=tf.extractfile("ST_counts/matrix.mtx")
    line=fh.readline()
    while line.startswith(b"%"): line=fh.readline()
    hdr=line.split(); assert int(hdr[0])==nG and int(hdr[1])==len(bc), ("MTX DIMS",hdr)
    tot=0
    for ch in pd.read_csv(fh,sep=" ",header=None,names=["g","s","v"],chunksize=20_000_000,dtype={"g":np.int32,"s":np.int32,"v":np.float64}):
        sp=sid[ch.s.values-1]; m=sp>=0
        P+=np.bincount((ch.g.values[m]-1)*nS+sp[m],weights=ch.v.values[m],minlength=nG*nS); tot+=len(ch)
        print(" entries",tot,round(time.time()-t0),"s",flush=True)
assert tot==int(hdr[2]), ("MTX ENTRY COUNT",tot,hdr[2])
P=P.reshape(nG,nS); lib=P.sum(0)   # library size on the FULL gene matrix, before any subsetting [method rule]
R("row 6 pseudobulk: entries",tot,"; per-GSM library size min",int(lib.min()),"median",int(np.median(lib)))
c0=gl[0].astype(str).str.split(".").str[0]; c1=gl[1].astype(str) if gl.shape[1]>1 else c0
ent=[ENS2E.get(a) or SYM2E.get(b.upper()) for a,b in zip(c0,c1)]
CPM=np.log2(P/np.maximum(lib,1)*1e6+1)
E=np.full((len(genes),nS),np.nan); acc={}
for i,e in enumerate(ent):
    if e is not None and e in GI: acc.setdefault(GI[e],[]).append(CPM[i])
for gi_,L in acc.items(): E[gi_]=np.median(np.vstack(L),axis=0)
sp_=pd.read_csv("GSE174647_ST_spatial_coord_annotations_spots.csv.gz",usecols=["id","donor_pseud"]).drop_duplicates()
d_of=dict(zip(sp_.id,sp_.donor_pseud)); don=[d_of.get(t,"NA") for t in title]
R("row 6 GSMs with no donor:",sum(d=="NA" for d in don))
units=sorted(set((d,g_) for d,g_ in zip(don,grp) if g_ in("case","control") and d!="NA"))
both=set(d for d,g_ in units if g_=="case")&set(d for d,g_ in units if g_=="control")
R(f"row 6 donors {len(set(don))}, units {len(units)}, donors in both arms {len(both)}")
Ea=np.column_stack([np.mean(E[:,[i for i,(d,g_) in enumerate(zip(don,grp)) if d==u and g_==gg]],axis=1) for u,gg in units])
build("6","GSE174647",Ea,[gg for u,gg in units],note=f"spots summed to {nS} GSMs, log2 CPM, mean to donor via donor_pseud")

In [ ]:
# CELL 6: re-freeze
out={"prereg_sha256":"7eee1779246f461b8595eeef8cffd17ee93ead40f6ed36b6af390cd421b7c9a2","addendum":"v1.1 + v1.2","k":150,
     "statistic":"control-2 pooled-sd t with 0.1*median floor","signatures":SIG,"drugs":F.to_dict(orient="records"),
     "row13":"NOT BUILT: no processed expression deposited (decision pending)"}
b=json.dumps(out,sort_keys=True).encode(); open("signatures_frozen_B1b.json","wb").write(b)
assert open("signatures_frozen_B1b.json","rb").read()==b, "FREEZE READ-BACK MISMATCH"
h=hashlib.sha256(b).hexdigest()
print("=== PHASE B1b REPORT BEGIN ===")
print("\n".join(REPORT))
for row in sorted(SIG): v=SIG[row]; print(f"SIG row {row}: {v['series']} case {v['n_case']} ctrl {v['n_ctrl']} usable {v['genes_usable']} up[:5] {v['up'][:5]} dn[:5] {v['dn'][:5]}")
print("FROZEN signatures_frozen_B1b.json sha256",h)
print("=== PHASE B1b REPORT END ===")